# 09 — Economic decision framework

Analyses for the *Agronomy for Sustainable Development* manuscript (`ASD_maize_article/`): probabilistic evaluation of the out-of-sample yield distributions (CRPS, PIT, coverage, conditional conformal), Shapley decomposition of margin risk, yield–price natural hedge, cost of simpler decision rules, bootstrap stability of the recommended date and break-even yield. Reproduces the margin chain of notebook 05 (checked against `risco_economico_municipio_data.csv`). Outputs: `resultados/framework/` and three figures in `figuras/`.

In [1]:
"""Economic decision framework analyses (prototype for notebook 09). Reproduces the margin chain of notebook 05."""
import numpy as np, pandas as pd
from pathlib import Path
from scipy.special import logsumexp
from scipy.stats import norm

RAIZ = Path.cwd().parent
DADOS, RES = RAIZ / "dados", RAIZ / "resultados"
OUT = RES / "framework"
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(42)
ORDEM = [f"{d:02d}/{m:02d}" for m in (1, 2, 3) for d in (5, 15, 25)]
ANOS = range(2000, 2026); N = 200; FUN = 0.015; COB = 0.65; RR = 2

prod = pd.read_parquet(RES / "produtividade_prevista_pr.parquet")
prod = prod[~prod.sem_historico_ibge]
custos = pd.read_csv(RES / "custos_milho_safrinha_real.csv").set_index("ano")
precos = pd.read_csv(RES / "precos_milho_pr_mensal_real.csv").set_index(["ano", "mes"])["preco_real_kg"]
risco = pd.read_csv(RES / "risco_economico_municipio_data.csv")
zarc = risco[["cod_ibge", "semeadura", "apto", "risco_zarc"]]
base = pd.read_parquet(RES / "base_treino_hibrido.parquet")
area = base.groupby("cod_ibge")["area_colhida_ha"].mean()

In [2]:
# ----------------------------------------------------------------- 1. probabilistic yield evaluation
oof = pd.read_parquet(RES / "previsoes_fora_amostra.parquet")


def crps_sample(y, samples):
    """CRPS of an ensemble forecast (rows) for observations y, energy form."""
    s = np.sort(samples, axis=1); m = s.shape[1]
    t1 = np.abs(s - y[:, None]).mean(1)
    w = (2 * np.arange(1, m + 1) - m - 1) / m ** 2
    return t1 - (s * w).sum(1)


def prob_eval(col, mondrian=False):
    e = (oof.log_y - oof[col]).to_numpy(); anos = oof.ano.to_numpy(); pred = oof[col].to_numpy()
    anom = pred - oof["A_media_historica"].to_numpy()  # predicted weather anomaly
    crps, pit, cov = np.empty(len(oof)), np.empty(len(oof)), np.empty(len(oof))
    for a in np.unique(anos):
        te, tr = anos == a, anos != a
        if mondrian:  # residual pool conditioned on tercile of the predicted anomaly (training-year cut points)
            q = np.quantile(anom[tr], [1 / 3, 2 / 3])
            btr, bte = np.digitize(anom[tr], q), np.digitize(anom[te], q)
        else:
            btr, bte = np.zeros(tr.sum(), int), np.zeros(te.sum(), int)
        idx_te = np.where(te)[0]
        for k in np.unique(bte):
            r = e[tr][btr == k]
            r = rng.choice(r, 400, replace=False) if r.size > 400 else r
            ii = idx_te[bte == k]
            ens = np.exp(pred[ii, None] + r[None, :])
            yo = oof.rendimento_kg_ha.to_numpy()[ii]
            crps[ii] = crps_sample(yo, ens)
            pit[ii] = (ens < yo[:, None]).mean(1)
            lo, hi = np.quantile(r, [0.1, 0.9])
            cov[ii] = (np.log(yo) >= pred[ii] + lo) & (np.log(yo) <= pred[ii] + hi)
    return crps, pit, cov


pe = {}
for name, col, mon in [("A climatology", "A_media_historica", False), ("L conformal", "L_ensemble_E_SVR", False),
                       ("L Mondrian", "L_ensemble_E_SVR", True)]:
    c, p, v = prob_eval(col, mon)
    pe[name] = pd.DataFrame({"ano": oof.ano, "crps": c, "pit": p, "cov": v})
tab = pd.DataFrame({k: {"CRPS": d.crps.mean(), "coverage80": d["cov"].mean(),
                        "cov_2020_22": d[d.ano.between(2020, 2022)]["cov"].mean(),
                        "cov_min_year": d.groupby("ano")["cov"].mean().min()} for k, d in pe.items()}).T
tab["CRPSS_vs_A"] = 1 - tab.CRPS / tab.loc["A climatology", "CRPS"]
print("\n== probabilistic evaluation ==\n", tab.round(3))
# bootstrap over years of the CRPSS of L vs A
ca = pe["A climatology"].groupby("ano").crps.sum(); cl = pe["L conformal"].groupby("ano").crps.sum()
yrs = ca.index.to_numpy(); bi = rng.integers(0, len(yrs), (2000, len(yrs)))
ss = 1 - cl.to_numpy()[bi].sum(1) / ca.to_numpy()[bi].sum(1)
print("CRPSS L vs A 95% CI:", np.percentile(ss, [2.5, 50, 97.5]).round(3))
pit_hist = np.histogram(pe["L conformal"].pit, bins=10, range=(0, 1))[0] / len(oof)
print("PIT histogram (L):", pit_hist.round(3))
tab.to_csv(OUT / "avaliacao_probabilistica.csv")
pd.DataFrame({k: d.groupby("ano")["cov"].mean() for k, d in pe.items()}).to_csv(OUT / "cobertura_por_ano.csv")
pd.Series(pit_hist).to_csv(OUT / "pit_L.csv")


== probabilistic evaluation ==
                   CRPS  coverage80  cov_2020_22  cov_min_year  CRPSS_vs_A
A climatology  692.837       0.781        0.635         0.234       0.000
L conformal    599.766       0.804        0.550         0.395       0.134
L Mondrian     610.007       0.781        0.602         0.520       0.120
CRPSS L vs A 95% CI: [-0.013  0.131  0.259]
PIT histogram (L): [0.096 0.098 0.108 0.095 0.097 0.103 0.1   0.099 0.106 0.099]


In [3]:
# ----------------------------------------------------------------- 2. margin simulation (as notebook 05)
df = prod[prod.ano.isin(ANOS)].merge(zarc, on=["cod_ibge", "semeadura"]).reset_index(drop=True)
venda = pd.to_datetime(df.data_maturacao) + pd.Timedelta(days=20)
df["p"] = precos.reindex(pd.MultiIndex.from_arrays([venda.dt.year, venda.dt.month])).to_numpy()
c = custos.reindex(df.ano)
df["cha"], df["ckg"] = c["custo_operacional_ha_fixo"].to_numpy(), c["custo_var_real_kg"].to_numpy()
det = pd.read_excel(DADOS / "custos.xlsx", sheet_name="so_a_base", header=None).iloc[5:, [1, 5, 11]]
det.columns = ["cultura", "item", "valor"]
m2 = det[det.cultura == 7].set_index("item")["valor"].apply(pd.to_numeric, errors="coerce").fillna(0)
VAR = ["Operação de máquinas e implementos", "Despesas de manutenção de benfeitorias", "Mão-de-obra temporária",
       "Sementes/Manivas", "Fertilizantes", "Agrotóxicos", "Despesas gerais", "Transporte externo",
       "Assistência técnica", "PROAGRO/SEGURO", "Juros"]
df["premio"] = m2["PROAGRO/SEGURO"] / m2[VAR].sum() * c["custo_var_real_ha"].to_numpy()
mu = np.log(df.rend_P50_kg_ha.to_numpy())
sig = (np.log(df.rend_P90_kg_ha) - np.log(df.rend_P10_kg_ha)).to_numpy() / (2 * norm.ppf(0.9))
key = [df.cod_ibge.to_numpy(), df.semeadura.to_numpy()]
# as notebook 05: insured yield by the PSR rule (municipal mean, same for every date), tau by mesoregion, common random numbers
JAN_ZARC = ["15/01", "25/01", "05/02", "15/02", "25/02", "05/03", "15/03"]
yexp = df.cod_ibge.map(df[df.semeadura.isin(JAN_ZARC)].groupby("cod_ibge").rend_P50_kg_ha.mean()).to_numpy()
mun_meso = pd.read_csv(DADOS / "municipios_pr.csv").set_index("cod_ibge")["mesorregiao"]
tau_meso = pd.read_csv(RES / "tau_mesorregiao.csv", index_col=0)["tau"]
TAU = df.cod_ibge.map(mun_meso).map(tau_meso).fillna(tau_meso["_state"]).to_numpy()
zk = df.groupby(["cod_ibge", "ano"]).ngroup().to_numpy()
Z = rng.standard_normal((zk.max() + 1, N))[zk]
Y = np.exp(mu[:, None] + np.sqrt(sig ** 2 + TAU ** 2)[:, None] * Z)
W = float(custos.loc[list(ANOS), "custo_total_real_ha"].mean()); ra = RR / W
apto = df.apto.to_numpy()[:, None]


def margin(Y, p, cha, ckg, ins=True):
    pseg = pd.Series(p).groupby(key).transform("mean").to_numpy()
    rec = p[:, None] * Y * (1 - FUN); cost = cha[:, None] + ckg[:, None] * Y
    if ins:
        ind = np.maximum(COB * yexp[:, None] - Y, 0) * pseg[:, None]
        return rec - cost + np.where(apto, ind, df.premio.to_numpy()[:, None])
    return rec - cost + df.premio.to_numpy()[:, None]


M = margin(Y, df.p.to_numpy(), df.cha.to_numpy(), df.ckg.to_numpy())
grp = df.groupby(["cod_ibge", "semeadura"]).indices


def summarise(M, label=None):
    rows = []
    for (cod, d), ix in grp.items():
        v = M[ix].ravel()
        rows.append({"cod_ibge": cod, "semeadura": d, "mean": v.mean(),
                     "CE": -(logsumexp(-ra * v) - np.log(v.size)) / ra, "ploss": (v < 0).mean()})
    return pd.DataFrame(rows)


s0 = summarise(M).merge(zarc, on=["cod_ibge", "semeadura"])
chk = s0.merge(risco[["cod_ibge", "semeadura", "CE_rr2"]], on=["cod_ibge", "semeadura"])
print("\nCE check vs notebook 05 (corr, mean diff):", round(chk.CE.corr(chk.CE_rr2), 4), round((chk.CE - chk.CE_rr2).mean(), 1))


CE check vs notebook 05 (corr, mean diff): 0.9993 -0.2


In [4]:
# ----------------------------------------------------------------- 3. risk decomposition (yield / price / cost)
pm = pd.Series(df.p).groupby(key).transform("mean").to_numpy()
chm = df.cha.mean(); ckm = df.ckg.mean()
Ymed = np.exp(pd.Series(mu).groupby(key).transform("mean").to_numpy())[:, None] * np.ones((1, N))
fac = {}
import itertools
src = {"Y": (Y, Ymed), "p": (df.p.to_numpy(), pm), "c": ((df.cha.to_numpy(), df.ckg.to_numpy()),
                                                       (np.full(len(df), chm), np.full(len(df), ckm)))}


def var_within(Mx):
    return np.mean([Mx[ix].var() for ix in grp.values()])


vcache = {}
for subset in itertools.product([0, 1], repeat=3):  # 1 = factor varies
    Yx = src["Y"][0] if subset[0] else src["Y"][1]
    px = src["p"][0] if subset[1] else src["p"][1]
    cx = src["c"][0] if subset[2] else src["c"][1]
    vcache[subset] = var_within(margin(Yx, px, cx[0], cx[1], ins=False))
from math import factorial
shap = {}
for i, nm in enumerate(["yield", "price", "cost"]):
    tot = 0
    for subset in itertools.product([0, 1], repeat=3):
        if subset[i]: continue
        k = sum(subset); w = factorial(k) * factorial(3 - k - 1) / factorial(3)
        s1 = list(subset); s1[i] = 1
        tot += w * (vcache[tuple(s1)] - vcache[subset])
    shap[nm] = tot
vt = vcache[(1, 1, 1)]
print("\n== margin variance decomposition (no insurance, within municipality-date) ==")
print({k: round(v / vt, 3) for k, v in shap.items()}, "total SD R$/ha", round(np.sqrt(vt)))
pd.Series({k: v / vt for k, v in shap.items()}).to_csv(OUT / "decomposicao_risco.csv")

# natural hedge: annual state yield anomaly vs price of sale
st = df[df.semeadura.isin(["15/01", "05/02", "25/02", "15/03"])].groupby("ano")[["rend_P50_kg_ha", "p"]].mean()
print("corr(annual predicted yield, price) =", round(st.rend_P50_kg_ha.corr(st.p), 3))
obs = base.groupby("ano").rendimento_kg_ha.mean()
pr_ann = precos.groupby(level=0).mean().reindex(obs.index)
print("corr(observed state yield, mean annual price) =", round(obs.corr(pr_ann), 3))
# CE with prices paired vs prices shuffled across years (breaks the yield-price dependence)
anos_u = np.array(sorted(df.ano.unique())); perm = dict(zip(anos_u, rng.permutation(anos_u)))
pshuf = np.zeros(len(df)); hedge = []
for rep in range(20):
    perm = dict(zip(anos_u, rng.permutation(anos_u)))
    dd = df[["cod_ibge", "semeadura", "ano"]].assign(a2=df.ano.map(perm))
    lk = df.set_index(["cod_ibge", "semeadura", "ano"]).p
    pshuf = lk.reindex(pd.MultiIndex.from_frame(dd[["cod_ibge", "semeadura", "a2"]])).to_numpy()
    hedge.append(summarise(margin(Y, pshuf, df.cha.to_numpy(), df.ckg.to_numpy())).CE.mean())
print("mean CE paired", round(s0.CE.mean()), "| shuffled prices", round(np.mean(hedge)), "±", round(np.std(hedge)))


== margin variance decomposition (no insurance, within municipality-date) ==
{'yield': np.float64(0.794), 'price': np.float64(0.205), 'cost': np.float64(0.0)} total SD R$/ha 2885
corr(annual predicted yield, price) = -0.45
corr(observed state yield, mean annual price) = -0.12


mean CE paired 532 | shuffled prices 387 ± 71


In [5]:
# ----------------------------------------------------------------- 4. cost of ignoring economics
rp = risco.merge(prod[prod.ano.isin(ANOS)].groupby(["cod_ibge", "semeadura"]).rend_P50_kg_ha.mean().rename("yld").reset_index(),
                 on=["cod_ibge", "semeadura"])
ok = rp[rp.apto]
crit = {
    "max CE (rr=2)": ok.loc[ok.groupby("cod_ibge").CE_rr2.idxmax()],
    "max mean yield": ok.loc[ok.groupby("cod_ibge").yld.idxmax()],
    "lowest ZARC risk (earliest)": ok.sort_values(["cod_ibge", "risco_zarc"]).groupby("cod_ibge").head(1),
    "max mean margin (risk neutral)": ok.loc[ok.groupby("cod_ibge").margem_media.idxmax()],
    "uniform window mix": None,
}
ref = crit["max CE (rr=2)"].set_index("cod_ibge")
rows = []
for nm, sel in crit.items():
    if sel is None:
        w = ok[ok.semeadura.isin(["15/01", "05/02", "25/02", "15/03"])].groupby("cod_ibge")[["CE_rr2", "p_prejuizo"]].mean()
        ce, pl, same = w.CE_rr2, w.p_prejuizo, np.nan
    else:
        sel = sel.set_index("cod_ibge"); ce, pl = sel.CE_rr2, sel.p_prejuizo
        same = (sel.semeadura == ref.semeadura.reindex(sel.index)).mean()
    loss = (ref.CE_rr2 - ce.reindex(ref.index))
    a = area.reindex(loss.index).fillna(0)
    rows.append({"criterion": nm, "same_date_%": 100 * same, "CE_loss_mean_R$ha": loss.mean(),
                 "CE_loss_P90_R$ha": loss.quantile(0.9), "ploss": pl.mean(),
                 "state_CE_loss_R$M_yr": (loss * a).sum() / 1e6})
cost_ign = pd.DataFrame(rows).set_index("criterion")
print("\n== cost of ignoring economics/risk ==\n", cost_ign.round(2))
cost_ign.to_csv(OUT / "custo_ignorar_economia.csv")


== cost of ignoring economics/risk ==
                                 same_date_%  CE_loss_mean_R$ha  \
criterion                                                        
max CE (rr=2)                        100.00               0.00   
max mean yield                        94.35               2.55   
lowest ZARC risk (earliest)           86.02              41.37   
max mean margin (risk neutral)        95.70               1.20   
uniform window mix                      NaN             379.68   

                                CE_loss_P90_R$ha  ploss  state_CE_loss_R$M_yr  
criterion                                                                      
max CE (rr=2)                               0.00   0.25                  0.00  
max mean yield                              0.00   0.25                  7.81  
lowest ZARC risk (earliest)               181.60   0.26                 61.86  
max mean margin (risk neutral)              0.00   0.25                  5.71  
uniform window mi

In [6]:
# ----------------------------------------------------------------- 5. decision stability (bootstrap of years)
yr = df.ano.to_numpy(); yi = np.searchsorted(anos_u, yr)
gk = df.cod_ibge.astype(str) + "|" + df.semeadura
gcodes, ginv = np.unique(gk, return_inverse=True)
G = len(gcodes); Ty = len(anos_u)
# per (group, year): log-mean-exp of -ra*M over the draws, and the mean margin
LME = np.full((G, Ty), np.nan); MEAN = np.full((G, Ty), np.nan)
lme_row = logsumexp(-ra * M, axis=1) - np.log(N)
LME[ginv, yi] = lme_row; MEAN[ginv, yi] = M.mean(1)
gmun = np.array([g.split("|")[0] for g in gcodes]).astype(int); gdate = np.array([g.split("|")[1] for g in gcodes])
aptog = zarc.assign(k=zarc.cod_ibge.astype(str) + "|" + zarc.semeadura).set_index("k").apto.reindex(gcodes).to_numpy()
B = 1000; bidx = rng.integers(0, Ty, (B, Ty))
best_count = {}
muns = np.unique(gmun)
ce_b = -(logsumexp(LME[:, bidx], axis=2) - np.log(Ty)) / ra  # G x B
ce_b[~aptog] = -np.inf
prob = []
for m in muns:
    ii = np.where(gmun == m)[0]
    if not aptog[ii].any(): continue
    win = ii[np.argmax(ce_b[ii], axis=0)]
    vc = pd.Series(gdate[win]).value_counts(normalize=True)
    chosen = ref.semeadura.get(m)
    prob.append({"cod_ibge": m, "best": chosen, "p_best": vc.get(chosen, 0), "modal": vc.index[0], "p_modal": vc.iloc[0]})
prob = pd.DataFrame(prob)
print("\n== stability (bootstrap of 26 years, B=1000) ==")
print("median P(best date stays best):", round(prob.p_best.median(), 3),
      "| share with P>=0.8:", round((prob.p_best >= 0.8).mean(), 3), "| share with P<0.5:", round((prob.p_best < 0.5).mean(), 3))
print(prob.groupby("best").p_best.describe().round(2))
prob.to_csv(OUT / "estabilidade_bootstrap.csv", index=False)

# break-even yield by date (state mean)
be = df.assign(be=df.cha / (df.p * (1 - FUN) - df.ckg)).groupby("semeadura").agg(be=("be", "mean"), y=("rend_P50_kg_ha", "mean"))
be["safety_margin_%"] = 100 * (be.y / be.be - 1)
print("\n== break-even ==\n", be.loc[ORDEM].round(0))
be.to_csv(OUT / "equilibrio.csv")


== stability (bootstrap of 26 years, B=1000) ==
median P(best date stays best): 0.936 | share with P>=0.8: 0.723 | share with P<0.5: 0.013
       count  mean   std   min   25%   50%   75%   max
best                                                  
05/01  359.0  0.88  0.14  0.43  0.80  0.94  0.99  1.00
15/01   11.0  0.55  0.05  0.45  0.51  0.54  0.58  0.63
15/03    2.0  0.93  0.03  0.91  0.92  0.93  0.94  0.95

== break-even ==
                be       y  safety_margin_%
semeadura                                 
05/01      4220.0  6033.0             43.0
15/01      4263.0  5791.0             36.0
25/01      4285.0  5506.0             28.0
05/02      4322.0  5204.0             20.0
15/02      4348.0  4976.0             14.0
25/02      4379.0  4879.0             11.0
05/03      4376.0  4924.0             13.0
15/03      4352.0  5002.0             15.0
25/03      4320.0  5094.0             18.0


## Figures

In [7]:
import shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

RAIZ = Path.cwd().parent
import mapas
FW = RAIZ / "resultados" / "framework"
FIG = RAIZ / "figuras"
plt.rcParams.update({"font.size": 14})
ORDEM = [f"{d:02d}/{m:02d}" for m in (1, 2, 3) for d in (5, 15, 25)]

# ---------------- Fig 1: framework
fig, ax = plt.subplots(figsize=(15, 6.2)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 6.2)
boxes = [
    (0.2, 3.6, "Weather & soil\nBR-DWGD daily, 1999 to 2025\nAWC over cropland", "#dbe9f6"),
    (3.2, 3.6, "Crop model (AEZ-FAO)\n9 sowing dates × 27 seasons\nwater, frost, heat by phase", "#dbe9f6"),
    (6.2, 3.6, "Hybrid ML (AEZ features)\n+ potential-yield term\nby sowing date", "#e5f2e1"),
    (9.2, 3.6, "Yield distribution\nconformal quantiles\n+ field deviation τ (PSR)", "#e5f2e1"),
    (12.2, 3.6, "Margin distribution\npaired price & cost\ninsurance if ZARC-suitable", "#fbe6d4"),
    (12.2, 0.5, "Decision\ncertainty equivalent (SERF)\nP(loss), CVaR, break-even", "#fbe6d4"),
    (9.2, 0.5, "System & market\nsoybean break-even,\nforward pricing", "#f3e0ef"),
    (6.2, 0.5, "Independent checks\ninsurance claims by\nsowing date (PSR), IBGE", "#f3e0ef"),
    (3.2, 0.5, "Recommendation\nbest date, frontier,\ngap vs current calendar", "#f3e0ef"),
]
for x, y, t, c in boxes:
    ax.add_patch(FancyBboxPatch((x, y), 2.6, 2.0, boxstyle="round,pad=0.08", fc=c, ec="#555555", lw=1))
    ax.text(x + 1.3, y + 1.0, t, ha="center", va="center", fontsize=12.5)
arr = dict(arrowstyle="-|>", color="#333333", lw=1.5)
for x in (2.8, 5.8, 8.8, 11.8):
    ax.annotate("", (x + 0.4, 4.6), (x, 4.6), arrowprops=arr)
ax.annotate("", (13.5, 2.6), (13.5, 3.5), arrowprops=arr)
for x in (12.1, 9.1, 6.1):
    ax.annotate("", (x - 0.3, 1.5), (x, 1.5), arrowprops=arr)
ax.text(1.5, 5.95, "Agronomic layer", fontsize=13, weight="bold", color="#1f4e79")
ax.text(7.5, 5.95, "Statistical layer", fontsize=13, weight="bold", color="#2e6b2e")
ax.text(12.3, 5.95, "Economic layer", fontsize=13, weight="bold", color="#9c4a12")
plt.tight_layout(); fig.savefig(FIG / "fig_framework.png", dpi=300); plt.close(fig)

# ---------------- Fig: probabilistic evaluation
pit = pd.read_csv(FW / "pit_L.csv", index_col=0).iloc[:, 0].to_numpy()
cov = pd.read_csv(FW / "cobertura_por_ano.csv", index_col=0)
fig, axs = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 2.3]})
axs[0].bar(np.arange(0.05, 1, 0.1), pit, width=0.09, color="#4c78a8")
axs[0].axhline(0.1, color="k", ls=":"); axs[0].set(xlabel="PIT", ylabel="Fraction", ylim=(0, 0.15), title="(a) PIT, final model")
for col, st in zip(cov.columns, ["s-", "o-", "^-"]):
    axs[1].plot(cov.index, cov[col], st, label={"A climatology": "Historical mean (A)", "L conformal": "Final model, conformal",
                                                "L Mondrian": "Final model, conditional conformal"}[col], ms=5)
axs[1].axhline(0.8, color="k", ls=":"); axs[1].set(xlabel="Harvest year", ylabel="Coverage of P10-P90", ylim=(0, 1),
                                                  title="(b) Coverage of the 80% interval by held-out season")
axs[1].legend(frameon=False, fontsize=12, loc="lower left")
plt.tight_layout(); fig.savefig(FIG / "fig_avaliacao_probabilistica.png", dpi=300); plt.close(fig)

# ---------------- Fig: economic framework results
dec = pd.read_csv(FW / "decomposicao_risco.csv", index_col=0).iloc[:, 0]
be = pd.read_csv(FW / "equilibrio.csv", index_col=0).loc[ORDEM]
st = pd.read_csv(FW / "estabilidade_bootstrap.csv").set_index("cod_ibge")
risco = pd.read_csv(RAIZ / "resultados/risco_economico_municipio_data.csv")
prod = pd.read_parquet(RAIZ / "resultados/produtividade_prevista_pr.parquet")
yld = prod[prod.ano.between(2000, 2025)].groupby(["cod_ibge", "semeadura"]).rend_P50_kg_ha.mean().rename("yld").reset_index()
ok = risco.merge(yld, on=["cod_ibge", "semeadura"]); ok = ok[ok.apto]
ref = ok.loc[ok.groupby("cod_ibge").CE_rr2.idxmax()].set_index("cod_ibge").CE_rr2
crit = {"Max.\nyield": ok.loc[ok.groupby("cod_ibge").yld.idxmax()].set_index("cod_ibge").CE_rr2,
        "Lowest\nZARC risk": ok.sort_values(["cod_ibge", "risco_zarc"]).groupby("cod_ibge").head(1).set_index("cod_ibge").CE_rr2,
        "Risk\nneutral": ok.loc[ok.groupby("cod_ibge").margem_media.idxmax()].set_index("cod_ibge").CE_rr2,
        "Uniform\nmix": ok[ok.semeadura.isin(["15/01", "05/02", "25/02", "15/03"])].groupby("cod_ibge").CE_rr2.mean()}
fig = plt.figure(figsize=(18, 12))
a1 = fig.add_subplot(2, 3, 1); a2 = fig.add_subplot(2, 3, 2); a3 = fig.add_subplot(2, 3, 3); a4 = fig.add_subplot(2, 2, 3); a5 = fig.add_subplot(2, 2, 4)
a1.bar(["Yield", "Price", "Cost"], 100 * dec.reindex(["yield", "price", "cost"]).to_numpy(), color=["#4c78a8", "#f58518", "#9d9d9d"])
a1.set(ylabel="Share of margin variance (%)", title="(a) Sources of margin risk")
loss = [(ref - v.reindex(ref.index)).dropna() for v in crit.values()]
mn = np.array([l.mean() for l in loss]); p90 = np.array([l.quantile(0.9) for l in loss]); diff = [100 * (l > 0.5).mean() for l in loss]
a2.bar(range(4), mn, color="#54a24b"); a2.scatter(range(4), p90, marker="_", s=400, color="k", label="90th percentile")
for i in range(4):
    a2.text(i, max(mn[i], p90[i]) * 1.15 + 1, f"{diff[i]:.0f}% mun.\nchange", ha="center", fontsize=10.5)
a2.set_yscale("symlog", linthresh=10); a2.set_ylim(0, 3000)
a2.set_xticks(range(4), list(crit), fontsize=11.5); a2.legend(frameon=False, fontsize=11, loc="upper left")
a2.set(ylabel=r"CE lost vs. optimal date (R\$ ha$^{-1}$)", title="(b) Cost of other decision rules")
a3.plot(range(9), be.be, "o-", label="Break-even yield"); a3.plot(range(9), be.y, "s-", label="Median predicted yield")
a3.set_xticks(range(9), ORDEM, rotation=45, fontsize=11); a3.set(ylabel=r"kg ha$^{-1}$", title="(c) Break-even and expected yield"); a3.legend(frameon=False, fontsize=11)
sm = mapas.mapa(a4, st.p_best, cmap="RdYlGn", vmin=0, vmax=1, titulo="(d) Probability that the recommended date remains best\n(bootstrap of 26 seasons)")
fig.colorbar(sm, ax=a4, shrink=0.7)
cores = plt.cm.viridis(np.linspace(0, 0.95, 9)); cats = [d for d in ORDEM if d in set(st.best)]
mapas.mapa(a5, st.best, categorias=ORDEM, cores=list(cores), titulo="(e) Recommended date (CE, rr = 2)")
for d in cats:
    a5.scatter([], [], color=cores[ORDEM.index(d)], marker="s", label=d)
a5.legend(frameon=False, loc="upper right", fontsize=13, title="Sowing date")
fig.subplots_adjust(left=0.06, right=0.97, top=0.94, bottom=0.04, wspace=0.42, hspace=0.32); fig.savefig(FIG / "fig_framework_economico.png", dpi=300); plt.close(fig)
print("ok")

for f in ["fig_framework.png", "fig_avaliacao_probabilistica.png", "fig_framework_economico.png"]:
    shutil.copy(FIG / f, RAIZ.parent / "artigo_elsevier" / "figuras" / f)

ok
